# Feature Engineering

## Purpose
Create 12 new HR-specific features that capture
signals the raw data alone cannot express.

## Input
data/processed/employee_processed.csv

## Output
data/processed/employee_features.csv
(original 38 columns + 12 new engineered columns)

## Features Built
1. PerformanceTrend
2. SelfOtherGapBucket
3. OKREfficiencyScore
4. Rating360Variance
5. WorkloadRatio
6. TenureBucket
7. PromotionLagFlag
8. BurnoutWorkloadIndex
9. TrainingEfficiency
10. LeadershipGap
11. PromotionReadinessScore
12. EngagementPerfAlignment

In [1]:
# ── Imports and Load Data ──

import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings('ignore')

# ── File paths ──
notebook_dir   = os.getcwd()
project_root   = os.path.dirname(notebook_dir)

input_path  = os.path.join(project_root, 'data', 'processed',
                            'employee_processed.csv')
output_path = os.path.join(project_root, 'data', 'processed',
                            'employee_features.csv')

# ── Load data ──
df = pd.read_csv(input_path)

print(f"Data loaded successfully")
print(f"Rows    : {df.shape[0]:,}")
print(f"Columns : {df.shape[1]}")
print(f"\nColumns available:")
for i, col in enumerate(df.columns, 1):
    print(f"{i:>2}. {col}") 

Data loaded successfully
Rows    : 2,600
Columns : 38

Columns available:
 1. EmployeeID
 2. Department
 3. JobRole
 4. JobLevel
 5. ManagerID
 6. LocationType
 7. Age
 8. Gender
 9. EducationLevel
10. YearsAtCompany
11. YearsSinceLastPromotion
12. PerformanceRating
13. HistoricalRatingAvg
14. CalibrationAdjustedFlag
15. SelfRating
16. PeerAvgRating
17. SubordinateAvgRating
18. Overall360Score
19. SelfOtherGap
20. Leadership360
21. Collaboration360
22. OKRCompletionPct
23. NumOKRsAssigned
24. WeightedGoalAttainment
25. EngagementScore
26. JobSatisfaction
27. WorkLifeBalanceRating
28. BurnoutRisk
29. TrainingHoursLastYear
30. OvertimeHoursMonthly
31. AbsenteeismDays
32. AvgMonthlyHours
33. ProjectsHandled
34. PercentSalaryHikeLast
35. BonusPayoutPct
36. HighPotentialFlag
37. PIPHistoryFlag
38. HighPerformer


In [2]:
# ── Helper Function ──

def normalize_0_to_1(series):
    """
    Converts any numeric series to a 0-1 range.

    Example:
    Input:  [10, 20, 30, 40, 50]
    Output: [0.0, 0.25, 0.5, 0.75, 1.0]

    Used in PromotionReadinessScore to combine
    different metrics fairly.
    """
    min_val = series.min()
    max_val = series.max()

    # Avoid division by zero if all values are the same
    if max_val - min_val == 0:
        return pd.Series(np.zeros(len(series)), index=series.index)

    return (series - min_val) / (max_val - min_val)


print("Helper function defined")
print()
print("normalize_0_to_1 test:")
test = pd.Series([10, 20, 30, 40, 50])
result = normalize_0_to_1(test)
for original, normalized in zip(test, result):
    print(f"{original} → {normalized:.2f}") 

Helper function defined

normalize_0_to_1 test:
10 → 0.00
20 → 0.25
30 → 0.50
40 → 0.75
50 → 1.00


In [3]:
# ── Feature 1 — Performance Trend ──

# Formula: Current Rating - Historical Average Rating
df['PerformanceTrend'] = (
    df['PerformanceRating'] - df['HistoricalRatingAvg']
).round(2)

# ── Summary ──
print("Feature 1: PerformanceTrend created")
print()
print("Business meaning:")
print("Positive = improving vs last cycle")
print("Zero     = stable performance")
print("Negative = declining vs last cycle")
print()
print("Distribution:")
print(f"Min      : {df['PerformanceTrend'].min():.2f}")
print(f"Max      : {df['PerformanceTrend'].max():.2f}")
print(f"Mean     : {df['PerformanceTrend'].mean():.2f}")
print()

# Count improving vs declining
improving = (df['PerformanceTrend'] > 0.1).sum()
stable    = (df['PerformanceTrend'].between(-0.1, 0.1)).sum()
declining = (df['PerformanceTrend'] < -0.1).sum()

print(f"Improving  (trend > +0.1)  : {improving:,} employees "
      f"({improving/len(df)*100:.1f}%)")
print(f"Stable  (-0.1 to +0.1)     : {stable:,} employees "
      f"({stable/len(df)*100:.1f}%)")
print(f"Declining  (trend < -0.1)  : {declining:,} employees "
      f"({declining/len(df)*100:.1f}%)")
print()
print("HR Action:")
print("→ Declining High Performers = urgent retention/coaching flag")
print("→ Improving Low Performers  = coaching is working, continue") 

Feature 1: PerformanceTrend created

Business meaning:
Positive = improving vs last cycle
Zero     = stable performance
Negative = declining vs last cycle

Distribution:
Min      : -2.78
Max      : 1.50
Mean     : -0.02

Improving  (trend > +0.1)  : 996 employees (38.3%)
Stable  (-0.1 to +0.1)     : 602 employees (23.2%)
Declining  (trend < -0.1)  : 1,002 employees (38.5%)

HR Action:
→ Declining High Performers = urgent retention/coaching flag
→ Improving Low Performers  = coaching is working, continue


In [4]:
# ── Feature 2 — Self Other Gap Bucket ──

# Categorize the SelfOtherGap into meaningful labels
df['SelfOtherGapBucket'] = pd.cut(
    df['SelfOtherGap'],
    bins=[-99, -0.5, 0.5, 1.0, 99],
    labels=[
        'UnderEstimates Self',
        'Aligned',
        'Slight Inflation',
        'Significant Inflation'
    ]
)

# ── Summary ──
print("Feature 2: SelfOtherGapBucket created")
print()
print("Distribution:")
counts = df['SelfOtherGapBucket'].value_counts()
for category, count in counts.items():
    pct = count / len(df) * 100
    bar = "█" * int(pct / 2)
    print(f"{category:<25} {count:>5} ({pct:>5.1f}%) {bar}")
print()
print("HR Action:")
print("→ Significant Inflation = coaching on self-awareness")
print("→ UnderEstimates Self   = confidence building support")
print("→ Aligned               = healthy, no action needed") 

Feature 2: SelfOtherGapBucket created

Distribution:
Aligned                    1506 ( 57.9%) ████████████████████████████
Slight Inflation            709 ( 27.3%) █████████████
Significant Inflation       287 ( 11.0%) █████
UnderEstimates Self          98 (  3.8%) █

HR Action:
→ Significant Inflation = coaching on self-awareness
→ UnderEstimates Self   = confidence building support
→ Aligned               = healthy, no action needed


In [5]:
# ── Feature 3 — OKR Efficiency Score ──

# Formula: OKR Completion % / Number of OKRs Assigned
# Replace 0 with 1 in denominator to avoid division by zero
df['OKREfficiencyScore'] = (
    df['OKRCompletionPct'] /
    df['NumOKRsAssigned'].replace(0, 1)
).round(2)

# ── Summary ──
print("Feature 3: OKREfficiencyScore created")
print()
print("Formula: OKRCompletionPct / NumOKRsAssigned")
print()
print("Distribution:")
print(f"Min    : {df['OKREfficiencyScore'].min():.2f}")
print(f"Max    : {df['OKREfficiencyScore'].max():.2f}")
print(f"Mean   : {df['OKREfficiencyScore'].mean():.2f}")
print(f"Median : {df['OKREfficiencyScore'].median():.2f}")
print()

# Show example calculation
print("Example calculation:")
sample = df[['EmployeeID', 'OKRCompletionPct',
              'NumOKRsAssigned', 'OKREfficiencyScore']].head(5)
print(sample.to_string(index=False))
print()
print("HR Action:")
print("→ Low score + high OKRs = ambitious, needs support")
print("→ High score + low OKRs = may need stretch goals") 

Feature 3: OKREfficiencyScore created

Formula: OKRCompletionPct / NumOKRsAssigned

Distribution:
Min    : 1.25
Max    : 95.10
Mean   : 15.05
Median : 13.22

Example calculation:
EmployeeID  OKRCompletionPct  NumOKRsAssigned  OKREfficiencyScore
  EMP00001              42.3                7                6.04
  EMP00002              68.5                8                8.56
  EMP00003              72.6                6               12.10
  EMP00004              67.1                8                8.39
  EMP00005              50.7                4               12.68

HR Action:
→ Low score + high OKRs = ambitious, needs support
→ High score + low OKRs = may need stretch goals


In [6]:
# ── Feature 4 — Rating 360 Variance ──

# Standard deviation across the 3 rating sources
# skipna=True handles employees with no subordinate rating
rating_cols = ['SelfRating', 'PeerAvgRating', 'SubordinateAvgRating']

df['Rating360Variance'] = df[rating_cols].std(
    axis=1,
    skipna=True
).round(3)

# ── Summary ──
print("Feature 4: Rating360Variance created")
print()
print("Formula: std(SelfRating, PeerAvgRating, SubAvgRating)")
print()
print("Distribution:")
print(f"Min    : {df['Rating360Variance'].min():.3f}")
print(f"Max    : {df['Rating360Variance'].max():.3f}")
print(f"Mean   : {df['Rating360Variance'].mean():.3f}")
print()

# Flag high variance employees
high_var = (df['Rating360Variance'] > 1.0).sum()
print(f"High variance employees (std > 1.0) : {high_var} "
      f"({high_var/len(df)*100:.1f}%)")
print()
print("HR Action:")
print("→ High variance = schedule 360 debrief conversation")
print("→ Understand WHY different raters see them differently") 

Feature 4: Rating360Variance created

Formula: std(SelfRating, PeerAvgRating, SubAvgRating)

Distribution:
Min    : 0.000
Max    : 2.215
Mean   : 0.639

High variance employees (std > 1.0) : 435 (16.7%)

HR Action:
→ High variance = schedule 360 debrief conversation
→ Understand WHY different raters see them differently


In [7]:
# ── Feature 5 — Workload Ratio ──

# Expected monthly hours per job level
level_benchmarks = {
    'IC1'     : 160,
    'IC2'     : 168,
    'IC3'     : 175,
    'M1'      : 185,
    'M2'      : 195,
    'Director': 210
}

# Map each employee's level to their expected hours
df['LevelBenchmarkHours'] = df['JobLevel'].map(level_benchmarks)

# Handle any levels not in the dictionary
df['LevelBenchmarkHours'] = df['LevelBenchmarkHours'].fillna(170)

# Calculate ratio: actual / expected
df['WorkloadRatio'] = (
    df['AvgMonthlyHours'] / df['LevelBenchmarkHours']
).round(3)

# Remove helper column (not needed in final dataset)
df.drop(columns=['LevelBenchmarkHours'], inplace=True)

# ── Categorize workload zones ──
df['WorkloadZone'] = pd.cut(
    df['WorkloadRatio'],
    bins=[0, 0.9, 1.2, 1.5, 99],
    labels=['Under-Utilized', 'Optimal', 'Stretched', 'Overloaded']
)

# ── Summary ──
print("Feature 5: WorkloadRatio created")
print()
print("Formula: AvgMonthlyHours / LevelBenchmarkHours")
print()
print("Workload Zone Distribution:")
counts = df['WorkloadZone'].value_counts()
for zone, count in counts.sort_index().items():
    pct = count / len(df) * 100
    bar = "█" * int(pct / 2)
    print(f"{zone:<18} {count:>5} ({pct:>5.1f}%) {bar}")
print()
print("HR Action:")
print("→ Overloaded employees = workload review with manager")
print("→ Under-Utilized       = role enrichment conversation")
print("→ Optimal              = protect this balance") 

Feature 5: WorkloadRatio created

Formula: AvgMonthlyHours / LevelBenchmarkHours

Workload Zone Distribution:
Under-Utilized       102 (  3.9%) █
Optimal             2321 ( 89.3%) ████████████████████████████████████████████
Stretched            177 (  6.8%) ███
Overloaded             0 (  0.0%) 

HR Action:
→ Overloaded employees = workload review with manager
→ Under-Utilized       = role enrichment conversation
→ Optimal              = protect this balance


In [8]:
# ── Feature 6 — Tenure Bucket ──

df['TenureBucket'] = pd.cut(
    df['YearsAtCompany'],
    bins=[0, 1, 3, 7, 100],
    labels=['0-1yr', '1-3yr', '3-7yr', '7+yr']
)

# ── Summary ──
print("Feature 6: TenureBucket created")
print()
print("Distribution:")
counts = df['TenureBucket'].value_counts().sort_index()
for bucket, count in counts.items():
    pct = count / len(df) * 100
    bar = "█" * int(pct / 2)
    print(f"{bucket:<10} {count:>5} ({pct:>5.1f}%) {bar}")
print()

# Show average performance rating per tenure bucket
print("Average Performance Rating by Tenure Bucket:")
avg_rating = df.groupby('TenureBucket', observed=True)['PerformanceRating'].mean()
for bucket, avg in avg_rating.items():
    print(f"{bucket:<10} avg rating = {avg:.2f}")
print()
print("HR Action:")
print("→ 0-1yr with low rating = normal, support not PIP")
print("→ 7+yr with low rating  = role fit conversation needed") 

Feature 6: TenureBucket created

Distribution:
0-1yr        609 ( 23.4%) ███████████
1-3yr        798 ( 30.7%) ███████████████
3-7yr        756 ( 29.1%) ██████████████
7+yr         437 ( 16.8%) ████████

Average Performance Rating by Tenure Bucket:
0-1yr      avg rating = 3.00
1-3yr      avg rating = 3.04
3-7yr      avg rating = 2.94
7+yr       avg rating = 2.97

HR Action:
→ 0-1yr with low rating = normal, support not PIP
→ 7+yr with low rating  = role fit conversation needed


In [9]:
# ── Feature 7 — Promotion Lag Flag ──

# Flag = 1 if BOTH conditions are true:
# 1. Employee is High Potential (HighPotentialFlag = 1)
# 2. Has not been promoted in 3 or more years
df['PromotionLagFlag'] = (
    (df['HighPotentialFlag'] == 1) &
    (df['YearsSinceLastPromotion'] >= 3)
).astype(int)

# ── Summary ──
total_hipo      = df['HighPotentialFlag'].sum()
lagging         = df['PromotionLagFlag'].sum()
pct_of_hipo     = (lagging / total_hipo * 100) if total_hipo > 0 else 0

print("Feature 7: PromotionLagFlag created")
print()
print("Logic:")
print("Flag = 1 when HighPotential=1 AND YearsSincePromo >= 3")
print()
print(f"Total High Potential employees : {total_hipo:,}")
print(f"Of those — promotion lagging   : {lagging:,} "
      f"({pct_of_hipo:.1f}% of HiPos)")
print(f"% of total workforce           : "
      f"{lagging/len(df)*100:.1f}%")
print()

# Show their departments
if lagging > 0:
    print("Promotion Lag by Department:")
    lag_by_dept = (df[df['PromotionLagFlag'] == 1]
                   .groupby('Department')
                   .size()
                   .sort_values(ascending=False))
    for dept, count in lag_by_dept.items():
        print(f"{dept:<25} {count} employees")
print()
print("HR Action:")
print("→ These employees need promotion conversation NOW")
print("→ Flight risk if not addressed within 90 days") 

Feature 7: PromotionLagFlag created

Logic:
Flag = 1 when HighPotential=1 AND YearsSincePromo >= 3

Total High Potential employees : 276
Of those — promotion lagging   : 142 (51.4% of HiPos)
% of total workforce           : 5.5%

Promotion Lag by Department:
HR                        26 employees
Marketing                 26 employees
Sales                     24 employees
Customer Success          20 employees
Finance                   15 employees
Engineering               11 employees
Product                   11 employees
Operations                9 employees

HR Action:
→ These employees need promotion conversation NOW
→ Flight risk if not addressed within 90 days


In [10]:
# ── Feature 8 — Burnout Workload Index ──

# Convert BurnoutRisk text to number
burnout_map = {
    'Low'   : 0.2,
    'Medium': 0.6,
    'High'  : 1.0
}
burnout_numeric = df['BurnoutRisk'].map(burnout_map).fillna(0.5)

# Workload excess = how much over benchmark (0 if under)
workload_excess = (df['WorkloadRatio'] - 1).clip(lower=0)

# Normalize workload excess to 0-1
workload_excess_normalized = normalize_0_to_1(workload_excess)

# Combine: 60% subjective burnout + 40% objective workload
df['BurnoutWorkloadIndex'] = (
    0.6 * burnout_numeric +
    0.4 * workload_excess_normalized
).round(3)

# ── Summary ──
print("Feature 8: BurnoutWorkloadIndex created")
print()
print("Formula:")
print("60% × BurnoutRisk numeric + 40% × WorkloadExcess")
print()
print("Distribution:")
print(f"Min    : {df['BurnoutWorkloadIndex'].min():.3f}")
print(f"Max    : {df['BurnoutWorkloadIndex'].max():.3f}")
print(f"Mean   : {df['BurnoutWorkloadIndex'].mean():.3f}")
print()

# Flag critical employees
critical = (df['BurnoutWorkloadIndex'] > 0.7).sum()
print(f"Critical burnout risk (index > 0.7) : {critical} "
      f"({critical/len(df)*100:.1f}%)")
print()
print("HR Action:")
print("→ Index > 0.7 = immediate manager conversation")
print("→ Track monthly — rising index = early warning") 

Feature 8: BurnoutWorkloadIndex created

Formula:
60% × BurnoutRisk numeric + 40% × WorkloadExcess

Distribution:
Min    : 0.120
Max    : 1.000
Mean   : 0.373

Critical burnout risk (index > 0.7) : 54 (2.1%)

HR Action:
→ Index > 0.7 = immediate manager conversation
→ Track monthly — rising index = early warning


In [11]:
# ── Feature 9 — Training Efficiency ──

# Formula: PerformanceTrend / TrainingHours
# Replace 0 training hours with 1 to avoid division by zero
df['TrainingEfficiency'] = (
    df['PerformanceTrend'] /
    df['TrainingHoursLastYear'].replace(0, 1)
).round(4)

# ── Summary ──
print("Feature 9: TrainingEfficiency created")
print()
print("Formula: PerformanceTrend / TrainingHoursLastYear")
print()
print("Distribution:")
print(f"Min    : {df['TrainingEfficiency'].min():.4f}")
print(f"Max    : {df['TrainingEfficiency'].max():.4f}")
print(f"Mean   : {df['TrainingEfficiency'].mean():.4f}")
print()

# Positive vs negative training efficiency
positive = (df['TrainingEfficiency'] > 0).sum()
negative = (df['TrainingEfficiency'] < 0).sum()
print(f"Training helped  (positive) : {positive:,} "
      f"({positive/len(df)*100:.1f}%)")
print(f"Training no help (negative) : {negative:,} "
      f"({negative/len(df)*100:.1f}%)")
print()
print("HR Action (L&D ROI analysis):")
print("→ Positive group = training content is relevant")
print("→ Negative group = review training program alignment") 

Feature 9: TrainingEfficiency created

Formula: PerformanceTrend / TrainingHoursLastYear

Distribution:
Min    : -0.9700
Max    : 0.8700
Mean   : -0.0029

Training helped  (positive) : 1,265 (48.7%)
Training no help (negative) : 1,223 (47.0%)

HR Action (L&D ROI analysis):
→ Positive group = training content is relevant
→ Negative group = review training program alignment


In [12]:
# ── Feature 10 — Leadership Gap ──

# Fill missing Overall360Score with median before calculating
overall_360_filled = df['Overall360Score'].fillna(
    df['Overall360Score'].median()
)
leadership_filled = df['Leadership360'].fillna(
    df['Leadership360'].median()
)

# Formula: Leadership360 - Overall360Score
df['LeadershipGap'] = (
    leadership_filled - overall_360_filled
).round(2)

# ── Summary ──
print("Feature 10: LeadershipGap created")
print()
print("Formula: Leadership360 - Overall360Score")
print()
print("Distribution:")
print(f"Min    : {df['LeadershipGap'].min():.2f}")
print(f"Max    : {df['LeadershipGap'].max():.2f}")
print(f"Mean   : {df['LeadershipGap'].mean():.2f}")
print()

# Managers vs ICs
managers = df[df['JobLevel'].isin(['M1', 'M2', 'Director'])]
ics      = df[~df['JobLevel'].isin(['M1', 'M2', 'Director'])]

print(f"Average Leadership Gap:")
print(f"Managers   : {managers['LeadershipGap'].mean():.2f}")
print(f"ICs        : {ics['LeadershipGap'].mean():.2f}")
print()

# Development need flags
dev_needed = (
    (df['JobLevel'].isin(['M1', 'M2', 'Director'])) &
    (df['LeadershipGap'] < -0.5)
).sum()
print(f"Managers with leadership gap < -0.5 : {dev_needed}")
print()
print("HR Action:")
print("→ Negative gap managers = leadership development program")
print("→ Positive gap ICs      = future manager pipeline") 

Feature 10: LeadershipGap created

Formula: Leadership360 - Overall360Score

Distribution:
Min    : -3.62
Max    : 2.50
Mean   : -0.68

Average Leadership Gap:
Managers   : -0.51
ICs        : -0.74

Managers with leadership gap < -0.5 : 312

HR Action:
→ Negative gap managers = leadership development program
→ Positive gap ICs      = future manager pipeline


In [13]:
# ── Feature 11 — Promotion Readiness Score ──

# Fill missing values before normalizing
overall_360_clean = df['Overall360Score'].fillna(
    df['Overall360Score'].median()
)

# Each component normalized to 0-1 range
perf_norm    = normalize_0_to_1(df['PerformanceRating'])
score360norm = normalize_0_to_1(overall_360_clean)
okr_norm     = normalize_0_to_1(df['OKRCompletionPct'])
tenure_norm  = normalize_0_to_1(df['YearsAtCompany'].clip(0, 7))

# Recency: HIGHER years since promotion = MORE eligible
# (capped at 5 years)
promo_lag_norm = normalize_0_to_1(
    df['YearsSinceLastPromotion'].clip(0, 5)
)

# Weighted combination
df['PromotionReadinessScore'] = (
    0.30 * perf_norm      +   # Performance is most important
    0.20 * score360norm   +   # How others see them
    0.20 * okr_norm       +   # Goal attainment
    0.15 * df['HighPotentialFlag'] +  # Already identified HiPo
    0.10 * tenure_norm    +   # Enough experience
    0.05 * promo_lag_norm     # Eligible (not promoted recently)
).round(3)

# ── Categorize readiness ──
df['ReadinessCategory'] = pd.cut(
    df['PromotionReadinessScore'],
    bins=[0, 0.35, 0.50, 0.65, 0.75, 1.01],
    labels=[
        'Not Yet Ready',
        'Development Needed',
        'Ready in 12 Months',
        'Ready in 6 Months',
        'Ready Now'
    ]
)

# ── Summary ──
print("Feature 11: PromotionReadinessScore created")
print()
print("Weights:")
print("30% Performance Rating")
print("20% Overall 360 Score")
print("20% OKR Completion %")
print("15% High Potential Flag")
print("10% Years at Company (tenure)")
print("5% Years Since Last Promotion")
print()
print("Score Range: 0.0 (not ready) to 1.0 (fully ready)")
print(f"Min    : {df['PromotionReadinessScore'].min():.3f}")
print(f"Max    : {df['PromotionReadinessScore'].max():.3f}")
print(f"Mean   : {df['PromotionReadinessScore'].mean():.3f}")
print()
print("Readiness Category Distribution:")
counts = df['ReadinessCategory'].value_counts().sort_index()
for cat, count in counts.items():
    pct = count / len(df) * 100
    bar = "█" * int(pct / 2)
    print(f"{cat:<22} {count:>5} ({pct:>5.1f}%) {bar}")
print()
print("HR Action:")
print("→ Ready Now = start promotion process this cycle")
print("→ Ready in 6 Months = prepare promotion business case")
print("→ Not Yet Ready = build development plan") 

Feature 11: PromotionReadinessScore created

Weights:
30% Performance Rating
20% Overall 360 Score
20% OKR Completion %
15% High Potential Flag
10% Years at Company (tenure)
5% Years Since Last Promotion

Score Range: 0.0 (not ready) to 1.0 (fully ready)
Min    : 0.029
Max    : 1.000
Mean   : 0.468

Readiness Category Distribution:
Not Yet Ready            650 ( 25.0%) ████████████
Development Needed       968 ( 37.2%) ██████████████████
Ready in 12 Months       617 ( 23.7%) ███████████
Ready in 6 Months        149 (  5.7%) ██
Ready Now                216 (  8.3%) ████

HR Action:
→ Ready Now = start promotion process this cycle
→ Ready in 6 Months = prepare promotion business case
→ Not Yet Ready = build development plan


In [14]:
# ── Feature 12 — Engagement Performance Alignment ──

# Normalize both to 0-1 then multiply
# High value = high on BOTH dimensions
engagement_norm = normalize_0_to_1(df['EngagementScore'])
perf_norm_2     = normalize_0_to_1(df['PerformanceRating'])

df['EngagementPerfAlignment'] = (
    engagement_norm * perf_norm_2
).round(3)

# ── Create the 4 quadrants ──
eng_median  = df['EngagementScore'].median()
perf_median = df['PerformanceRating'].median()

conditions = [
    (df['EngagementScore'] >= eng_median) & (df['PerformanceRating'] >= perf_median),
    (df['EngagementScore'] >= eng_median) & (df['PerformanceRating'] <  perf_median),
    (df['EngagementScore'] <  eng_median) & (df['PerformanceRating'] >= perf_median),
    (df['EngagementScore'] <  eng_median) & (df['PerformanceRating'] <  perf_median),
]
quadrant_labels = [
    'Star (High Eng + High Perf)',
    'Coachable (High Eng + Low Perf)',
    'Flight Risk (Low Eng + High Perf)',
    'Needs Support (Low Eng + Low Perf)',
]

df['EngagementPerfQuadrant'] = np.select(
    conditions,
    quadrant_labels,
    default='Unknown'
)

# ── Summary ──
print("Feature 12: EngagementPerfAlignment created")
print()
print("Formula: normalize(Engagement) × normalize(Performance)")
print()
print("4-Quadrant Distribution:")
counts = df['EngagementPerfQuadrant'].value_counts()
for quadrant, count in counts.items():
    pct = count / len(df) * 100
    bar = "█" * int(pct / 2)
    print(f"{quadrant:<38} {count:>5} ({pct:>5.1f}%) {bar}")
print()
print("HR Action:")
print("→ Flight Risk employees = retention conversation THIS WEEK")
print("→ Star employees        = retention bonus / fast track")
print("→ Coachable employees   = invest in development plan") 

Feature 12: EngagementPerfAlignment created

Formula: normalize(Engagement) × normalize(Performance)

4-Quadrant Distribution:
Star (High Eng + High Perf)             1095 ( 42.1%) █████████████████████
Needs Support (Low Eng + Low Perf)       858 ( 33.0%) ████████████████
Flight Risk (Low Eng + High Perf)        437 ( 16.8%) ████████
Coachable (High Eng + Low Perf)          210 (  8.1%) ████

HR Action:
→ Flight Risk employees = retention conversation THIS WEEK
→ Star employees        = retention bonus / fast track
→ Coachable employees   = invest in development plan


In [15]:
# ── Final Summary and Save ──

# ── Show all 12 new features ──
new_features = [
    'PerformanceTrend',
    'SelfOtherGapBucket',
    'OKREfficiencyScore',
    'Rating360Variance',
    'WorkloadRatio',
    'WorkloadZone',
    'TenureBucket',
    'PromotionLagFlag',
    'BurnoutWorkloadIndex',
    'TrainingEfficiency',
    'LeadershipGap',
    'PromotionReadinessScore',
    'ReadinessCategory',
    'EngagementPerfAlignment',
    'EngagementPerfQuadrant',
]

print("-" * 55)
print("SUMMARY — ALL FEATURES BUILT")
print("-" * 55)
print()
print(f"Original columns   : 39")
print(f"New features added : {len(new_features)}")
print(f"Total columns now  : {df.shape[1]}")
print()
print("New features created:")
for i, feat in enumerate(new_features, 1):
    dtype = str(df[feat].dtype)
    missing = df[feat].isnull().sum()
    print(f"{i:>2}. {feat:<30} type={dtype:<15} missing={missing}")

# ── Save to CSV ──
df.to_csv(output_path, index=False)

print()
print(f"Feature-engineered dataset saved:")
print(f"{output_path}")
print()
print(f"Rows    : {df.shape[0]:,}")
print(f"Columns : {df.shape[1]}")
print()
print("-" * 55)
print("Ready for Exploratory Data Analysis")  
print("-" * 55) 

-------------------------------------------------------
SUMMARY — ALL FEATURES BUILT
-------------------------------------------------------

Original columns   : 39
New features added : 15
Total columns now  : 53

New features created:
 1. PerformanceTrend               type=float64         missing=0
 2. SelfOtherGapBucket             type=category        missing=0
 3. OKREfficiencyScore             type=float64         missing=0
 4. Rating360Variance              type=float64         missing=0
 5. WorkloadRatio                  type=float64         missing=0
 6. WorkloadZone                   type=category        missing=0
 7. TenureBucket                   type=category        missing=0
 8. PromotionLagFlag               type=int64           missing=0
 9. BurnoutWorkloadIndex           type=float64         missing=0
10. TrainingEfficiency             type=float64         missing=0
11. LeadershipGap                  type=float64         missing=0
12. PromotionReadinessScore        ty